# Упражнение: RAG: чанкинг с перекрытием, recall@k и MRR

## Задание

Мини-RAG без нейросетей: нарезаем документы на чанки, ищем чанки по запросу и оцениваем поиск.

`input.csv` — 8 документов (`doc_id, text`, слова разделены пробелом, по 65 слов). В каждом документе спрятан
«факт» из пяти слов вида `темаN имеет кодовое значение кодN`; вокруг — случайные слова-заполнители.
`queries.csv` — 16 запросов (`query_id, query, doc_id, answer`): в запросе слово `темаN` и два случайных слова,
`doc_id` — документ с ответом, `answer` — слово-ответ (`кодN`, в запросе его нет).

**Чанкинг.** Для документа из слов w₀…w₍ₘ₋₁₎ чанки — это срезы `words[i : i + size]` для
`i = 0, step, 2·step, …` пока `i < max(m − overlap, 1)`, где `step = size − overlap`. Последний чанк может быть
короче. Чанки нумеруются сквозным индексом: сначала все чанки документа 1, затем документа 2 и так далее.

**Релевантные чанки запроса** — чанки документа `doc_id`, в которых встречается слово `answer`
(при перекрытии таких чанков бывает два).

**Поиск.** Пусть N — общее число чанков, nₜ — число чанков, содержащих слово t.
Вес слова: IDF(t) = ln(1 + (N − nₜ + 0.5) / (nₜ + 0.5)). Оценка чанка для запроса — сумма IDF по **различным**
словам запроса, которые встречаются в чанке (повторы слов внутри чанка не учитываются). Чанки сортируются
по убыванию оценки; при равенстве — по возрастанию индекса чанка.

**Метрики** (усредняются по 16 запросам), K = 3:
- `recall_at_3` = (число релевантных чанков среди 3 лучших) / (число всех релевантных чанков запроса);
- `mrr` = среднее от 1 / rank, где rank — место первого релевантного чанка в полном ранжировании (с единицы).

Посчитайте метрики для четырёх конфигураций чанкинга: (size, overlap) = (10, 0), (10, 3), (20, 0), (20, 5).

## Подсказки

- Сначала только чанкинг: при size = 10 и overlap = 0 из документа в 65 слов получается 7 чанков, при overlap = 3
  шаг равен 7 — проверьте число чанков вручную.
- IDF считайте отдельно для каждой конфигурации (N и nₜ зависят от разбиения). Сумму по словам запроса ведите
  в одном и том же порядке (`sorted(set(query.split()))`) и округляйте оценку до 9 знаков — иначе шум
  плавающей точки сломает ничьи.
- Для каждого запроса: `order = sorted(range(N), key=lambda i: (-score[i], i))`, `relevant = {i : owner[i] == doc_id
  and answer in chunk[i]}`, `rank = первый номер в order, где i in relevant`.

## Ожидаемый формат выходного файла

CSV с колонками `size, overlap, n_chunks, recall_at_3, mrr` — 4 строки в порядке конфигураций из условия.
`n_chunks` — общее число чанков N, метрики округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/rag
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.